# AE2: manejo de archivos CSV, JSON y TXT

Flujo: descarga manual → importar → modificar → guardar nuevo CSV y JSON → tercera clase → resumen TXT con URL de origen.

Ejecuta las celdas en orden.

## 1. URL de origen y rutas de los archivos

Los archivos se descargaron manualmente de Hugging Face y se guardaron en `datos_originales/`. Las URL específicas se guardan en variables para enviarlas luego al resumen.

In [12]:
URL_CSV = "https://huggingface.co/datasets/fka/prompts.chat/blob/main/prompts.csv"
URL_JSON = "https://huggingface.co/datasets/leonorv/slaying/blob/main/slaying.json"

RUTA_CSV = "datos_originales/prompts.csv"
RUTA_JSON = "datos_originales/slaying.json"

RUTA_CSV_NUEVO = "datos_modificados/prompts_modificado.csv"
RUTA_JSON_NUEVO = "datos_modificados/datos_modificado.json"
RUTA_RESUMEN = "resumen.txt"

## 2. Importar las clases e inicializar los objetos

In [13]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTXT

csv_obj = ManejadorCSV(RUTA_CSV)
json_obj = ManejadorJSON(RUTA_JSON)


def corto(valor, maximo=60):
    
    texto = str(valor)
    return texto if len(texto) <= maximo else texto[:maximo] + "..."

## 3. Importar los archivos locales

In [14]:
csv_obj.importar()
json_obj.importar()

print("CSV  - columnas:", csv_obj.encabezados)
print("CSV  - registros importados:", csv_obj.registros_importados)
print("JSON - registros importados:", json_obj.registros_importados)
print("JSON - primer registro:", {k: corto(v) for k, v in json_obj.registros[0].items()})

CSV  - columnas: ['act', 'prompt', 'for_devs', 'type', 'contributor']
CSV  - registros importados: 2627
JSON - registros importados: 3405
JSON - primer registro: {'id': '214515', 'term': 'punks', 'definition': 'A slang term with multiple meanings depending on context. Or...', 'sentence': 'As anti establishment, punks have always mostly been on the ...', 'communities': "['transgender', 'gay male']", 'metadata': "{'source': 'Reddit', 'subreddit': 'transgender', 'date': '20..."}


## 4. Modificar el CSV (antes y después)

Con métodos de `ManejadorCSV` se **actualiza un valor**, se **agrega** un registro y se **elimina** otro.

In [15]:
columna = "act"   

print("ANTES  - primera fila:", [corto(c) for c in csv_obj.filas[0]])
print("ANTES  - total de filas:", len(csv_obj.filas))

# 1) Actualizar un valor
csv_obj.actualizar_valor(0, columna, csv_obj.filas[0][0] + " (editado)")

# 2) Agregar un registro nuevo
csv_obj.agregar_registro({
    "act": "Registro de prueba",
    "prompt": "Registro agregado desde el notebook.",
    "for_devs": "FALSE",
    "type": "TEXT",
    "contributor": "estudiante",
})

# 3) Eliminar un registro
csv_obj.eliminar_registro(1)

print("DESPUÉS - primera fila:", [corto(c) for c in csv_obj.filas[0]])
print("DESPUÉS - última fila:", [corto(c) for c in csv_obj.filas[-1]])
print("DESPUÉS - total de filas:", len(csv_obj.filas))

ANTES  - primera fila: ['Ethereum Developer', 'Imagine you are an experienced Ethereum developer tasked wit...', 'TRUE', 'TEXT', 'ameya-2003']
ANTES  - total de filas: 2627
DESPUÉS - primera fila: ['Ethereum Developer (editado)', 'Imagine you are an experienced Ethereum developer tasked wit...', 'TRUE', 'TEXT', 'ameya-2003']
DESPUÉS - última fila: ['Registro de prueba', 'Registro agregado desde el notebook.', 'FALSE', 'TEXT', 'estudiante']
DESPUÉS - total de filas: 2627


## 5. Modificar el JSON (antes y después)

In [16]:
campo = "term"

print("ANTES  - primer registro:", {k: corto(v) for k, v in json_obj.registros[0].items()})
print("ANTES  - total de registros:", len(json_obj.registros))

json_obj.actualizar_valor(0, campo, "VALOR_ACTUALIZADO")

nuevo_json = dict(json_obj.registros[0])
nuevo_json[campo] = "REGISTRO_NUEVO"
json_obj.agregar_registro(nuevo_json)

json_obj.eliminar_registro(1)

print("DESPUÉS - primer registro:", {k: corto(v) for k, v in json_obj.registros[0].items()})
print("DESPUÉS - último registro:", {k: corto(v) for k, v in json_obj.registros[-1].items()})
print("DESPUÉS - total de registros:", len(json_obj.registros))

ANTES  - primer registro: {'id': '214515', 'term': 'punks', 'definition': 'A slang term with multiple meanings depending on context. Or...', 'sentence': 'As anti establishment, punks have always mostly been on the ...', 'communities': "['transgender', 'gay male']", 'metadata': "{'source': 'Reddit', 'subreddit': 'transgender', 'date': '20..."}
ANTES  - total de registros: 3405
DESPUÉS - primer registro: {'id': '214515', 'term': 'VALOR_ACTUALIZADO', 'definition': 'A slang term with multiple meanings depending on context. Or...', 'sentence': 'As anti establishment, punks have always mostly been on the ...', 'communities': "['transgender', 'gay male']", 'metadata': "{'source': 'Reddit', 'subreddit': 'transgender', 'date': '20..."}
DESPUÉS - último registro: {'id': '214515', 'term': 'REGISTRO_NUEVO', 'definition': 'A slang term with multiple meanings depending on context. Or...', 'sentence': 'As anti establishment, punks have always mostly been on the ...', 'communities': "['transgender', '

## 6. Guardar los nuevos archivos y verificar los cambios

In [17]:
csv_obj.guardar(RUTA_CSV_NUEVO)
json_obj.guardar(RUTA_JSON_NUEVO)

# Verificación: se vuelven a leer los archivos guardados
verif_csv = ManejadorCSV(RUTA_CSV_NUEVO)
verif_csv.importar()
verif_json = ManejadorJSON(RUTA_JSON_NUEVO)
verif_json.importar()

print("CSV guardado  - registros:", verif_csv.registros_importados, "| primera fila:", [corto(c) for c in verif_csv.filas[0]])
print("JSON guardado - registros:", verif_json.registros_importados, "| primer registro:", {k: corto(v) for k, v in verif_json.registros[0].items()})

CSV guardado  - registros: 2627 | primera fila: ['Ethereum Developer (editado)', 'Imagine you are an experienced Ethereum developer tasked wit...', 'TRUE', 'TEXT', 'ameya-2003']
JSON guardado - registros: 3405 | primer registro: {'id': '214515', 'term': 'VALOR_ACTUALIZADO', 'definition': 'A slang term with multiple meanings depending on context. Or...', 'sentence': 'As anti establishment, punks have always mostly been on the ...', 'communities': "['transgender', 'gay male']", 'metadata': "{'source': 'Reddit', 'subreddit': 'transgender', 'date': '20..."}


## 7. Generar el resumen.txt con la tercera clase

In [18]:
generador = GeneradorResumenTXT(URL_CSV, URL_JSON)
ruta = generador.generar(csv_obj.resultados(), json_obj.resultados(), RUTA_RESUMEN)

print("Resumen guardado en:", ruta)
print()
print(generador.leer())

Resumen guardado en: resumen.txt

RESUMEN DE EJECUCIÓN - AE2
Generado: 2026-10-24 16:46:16
Clases: ManejadorCSV, ManejadorJSON, GeneradorResumenTXT
Registros procesados en total (importados): 6032

ARCHIVO CSV
------------------------------------------------------------
URL de origen: https://huggingface.co/datasets/fka/prompts.chat/blob/main/prompts.csv
Clase utilizada: ManejadorCSV
Métodos utilizados: importar, actualizar_valor, agregar_registro, eliminar_registro, guardar
Archivo importado: datos_originales/prompts.csv
Nuevo archivo guardado: datos_modificados/prompts_modificado.csv
Registros importados: 2627
Registros en el archivo guardado: 2627
Modificaciones realizadas:
  - Actualizado: fila 0, columna 'act': 'Ethereum Developer' -> 'Ethereum Developer (editado)'
  - Agregado: registro nuevo {'act': 'Registro de prueba', 'prompt': 'Registro agregado desde el notebook.', 'for_devs': 'FALSE', 'type': 'TEXT', 'contributor': 'estudiante'}
  - Eliminado: fila 1 ['Linux Terminal', 'I 

## 8. Transformaciones realizadas

- **CSV:** se actualizó un valor de la primera fila, se agregó un registro nuevo y se eliminó otro.
- **JSON:** se actualizó un campo del primer registro, se agregó un registro nuevo y se eliminó otro.

Los archivos originales se conservan en `datos_originales/`; las versiones modificadas están en `datos_modificados/`.